# Creating a simple dashboard for sharing results online

In [1]:
import os
import shutil
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

#### Get all related files

In [2]:
english_resid_path = 'english/individual_corpora'
english_exponential_difference_path = 'english/4-complexity_tests/Allan_Variance/self-other-comparisons/individual_corpora'

english_corpus_files = [
    f for f in os.listdir(english_resid_path) 
    if (not f.startswith('._')) and f.endswith('.html')
]

english_exp_plot_files = [
    f for f in os.listdir(english_exponential_difference_path) 
    if (not f.startswith('._')) and f.endswith('.html')
]

english_files = set(english_corpus_files).intersection(set(english_exp_plot_files))
english_files

{'CABNC.html',
 'CANDOR.html',
 'CORAAL.html',
 'DISPEL.html',
 'Frederiksen.html',
 'GCSAusE.html',
 'Graesser.html',
 'MICASE.html',
 'Miao-fNIRS.html',
 'SBCSAE.html',
 'SCoSE.html',
 'callfriend.html',
 'callhome.html',
 'med_school.html'}

In [3]:
multilingual_resid_path = 'multilingual/individual_corpora'
multilingual_exponential_difference_path = 'multilingual/3-complexity_tests/self-other-comparisons/individual_corpora'

multilingual_corpus_files = [
    f for f in os.listdir(multilingual_resid_path) 
    if (not f.startswith('._')) and f.endswith('.html')
]

multilingual_exp_plot_files = [
    f for f in os.listdir(multilingual_exponential_difference_path) 
    if (not f.startswith('._')) and f.endswith('.html')
]

multilingual_files = set(multilingual_corpus_files).intersection(set(multilingual_exp_plot_files))
multilingual_files

{'deu.html', 'eng.html', 'fra.html', 'kor.html', 'spa.html', 'zho.html'}

#### Create Dashboard Template

In [4]:
f = open('dashboard_template.html', 'r')
dashboard_template = f.read()
f.close()

#### Prepare location for outputs

In [5]:
dashboard_loc = 'dash'
if not os.path.exists(dashboard_loc):
    os.mkdir(dashboard_loc)
    os.mkdir(os.path.join(dashboard_loc, 'plots'))

## Create English Block

In [6]:
html_img_block_text = """
<div>
    <em style="font-size:1.5em;">[[plot_name]]</em>
    <iframe src="./plots/eng-resid-[[plot]]" style="width: 100%; height: 250px;" frameBorder="0"></iframe>
    <p>&nbsp;&nbsp;(a) Change in CE for "self" (orange) and "other" (blue) separately by turn (turn $\Delta$)</p>
    <iframe src="./plots/eng-delta-[[plot]]" style="width: 100%; height: 250px;" frameBorder="0"></iframe>
    <p>&nbsp;&nbsp;(b) Change in difference in CE values between "self" versus "other" measurements by turn (turn $\Delta$)</p>
    <br>
    <br>
    <hr>
</div>

"""

In [7]:
outputs = []

In [8]:
for trace in english_files:
    try:
        shutil.copy(
            os.path.join(english_resid_path, trace),
            os.path.join('dash', 'plots', 'eng-resid-'+trace)
        )
        
        shutil.copy(
            os.path.join(english_exponential_difference_path, trace),
            os.path.join('dash', 'plots', 'eng-delta-'+trace)
        )
        
        outputs += [str(html_img_block_text).replace('[[plot]]', trace).replace('[[plot_name]]', trace.replace('.html', ''))]
    except Exception:
        pass

In [9]:
dashboard_template = dashboard_template.replace('[[ind_content_eng]]', '\n'.join(outputs))

## Create Multilingual Block

In [10]:
html_img_block_text = """
<div>
    <em style="font-size:1.5em;">[[plot_name]]</em>
    <iframe src="./plots/multi-resid-[[plot]]" style="width: 100%; height: 250px;" frameBorder="0"></iframe>
    <p>&nbsp;&nbsp;(a) Change in CE for "self" (orange) and "other" (blue) separately by turn (turn $\Delta$)</p>
    <iframe src="./plots/multi-delta-[[plot]]" style="width: 100%; height: 250px;" frameBorder="0"></iframe>
    <p>&nbsp;&nbsp;(b) Change in difference in CE values between "self" versus "other" measurements by turn (turn $\Delta$)</p>
    <br>
    <br>
    <hr>
</div>

"""

In [11]:
outputs = []

In [12]:
for trace in multilingual_files:
    shutil.copy(
        os.path.join(multilingual_resid_path, trace),
        os.path.join('dash', 'plots', 'multi-resid-'+trace)
    )
    
    shutil.copy(
        os.path.join(multilingual_exponential_difference_path, trace),
        os.path.join('dash', 'plots', 'multi-delta-'+trace)
    )
    
    outputs += [str(html_img_block_text).replace('[[plot]]', trace).replace('[[plot_name]]', trace.replace('.html', ''))]

In [13]:
dashboard_template = dashboard_template.replace('[[ind_content_multiling]]', '\n'.join(outputs))

## Candor Data Set-Up

In [14]:
def repair_survey_var_names(x):
    elements = x.split('_')[1:]
    return ' '.join(elements[:-1])

In [15]:
dfparams = pd.read_csv('english/5-CANDOR_exploration/data/parameter-estimates.csv',names=['0', 'parameter', '(1)', '(2)'])
del dfparams['0']

dfaic = pd.read_csv('english/5-CANDOR_exploration/data/aic.csv')

In [16]:
sel = (dfparams['parameter'] == dfparams['(1)']) & (dfparams['parameter'] == dfparams['(2)'])

dfparams['parameter'].loc[sel] = [repair_survey_var_names(x) for x in dfparams['parameter'].loc[sel]]

dfparams['(1)'].loc[sel] = None
dfparams['(2)'].loc[sel] = None

In [17]:
dfparams.head()

,parameter,(1),(2)
0,developed joint perspective,None,None
1,Intercept,5.06e+00 (2.99e-03)***,5.06e+00 (3.03e-03)***
2,resid,1.30e-05 (2.13e-03),NaN
3,resid2,NaN,-1.13e-03 (2.35e-04)***
4,in,None,None


In [18]:
dashboard_template = dashboard_template.replace('[[candor_params]]', dfparams.to_html(index=False).replace('NaN', '').replace('None', '').replace('border="1"', ''))
dashboard_template = dashboard_template.replace('[[candor_aic]]', dfaic.to_html(index=False).replace('NaN', '').replace('None', '').replace('border="1"', ''))

## Save dashboard

In [19]:
f = open('dash/dashboard.html', 'w')
f.write(dashboard_template)
f.close()